# Imports

In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import datetime
import os, glob

import warnings

warnings.filterwarnings('ignore')

# Loading Data

In [2]:
data_dir = "MyData"

In [3]:
jsons = glob.glob(os.path.join(data_dir, "*.json"))
jsons

['MyData/endsong_0.json',
 'MyData/endsong_1.json',
 'MyData/endsong_2.json',
 'MyData/endvideo.json']

In [4]:
df0 = pd.read_json(jsons[0])
df1 = pd.read_json(jsons[1])
df2 = pd.read_json(jsons[2])
df3 = pd.read_json(jsons[3])

In [5]:
df0.head(2).T

,0,1
ts,2022-02-25T06:11:22Z,2022-02-15T09:56:23Z
username,392uowv8fh00b9ie6ccn9cegs,392uowv8fh00b9ie6ccn9cegs
platform,"Android OS 10 API 29 (Xiaomi, Redmi Note 7S)","iOS 15.1 (iPad7,5)"
ms_played,238866,211160
conn_country,IN,IN
ip_addr_decrypted,47.15.125.198,202.142.121.72
user_agent_decrypted,unknown,unknown
master_metadata_track_name,Begin Again (Taylor's Version),Oops!...I Did It Again
master_metadata_album_artist_name,Taylor Swift,Britney Spears
master_metadata_album_album_name,Red (Taylor's Version),Oops!... I Did It Again


In [6]:
df1.head(2).T

,0,1
ts,2022-07-10T10:44:19Z,2021-01-21T08:27:48Z
username,392uowv8fh00b9ie6ccn9cegs,392uowv8fh00b9ie6ccn9cegs
platform,"Android OS 10 API 29 (Xiaomi, Redmi Note 7S)","Android OS 10 API 29 (Xiaomi, Redmi Note 7S)"
ms_played,210251,129633
conn_country,IN,IN
ip_addr_decrypted,103.27.9.101,117.211.192.30
user_agent_decrypted,unknown,unknown
master_metadata_track_name,the 1,Señorita
master_metadata_album_artist_name,Taylor Swift,Shawn Mendes
master_metadata_album_album_name,folklore,Shawn Mendes


In [7]:
df2.head(2).T

,0,1
ts,2021-11-12T08:11:23Z,2021-10-27T06:11:21Z
username,392uowv8fh00b9ie6ccn9cegs,392uowv8fh00b9ie6ccn9cegs
platform,"iOS 15.0.1 (iPad7,5)","Android OS 10 API 29 (Xiaomi, Redmi Note 7S)"
ms_played,136695,51292
conn_country,IN,IN
ip_addr_decrypted,202.142.122.89,203.81.240.198
user_agent_decrypted,unknown,unknown
master_metadata_track_name,Aman Syndai (Dragon Reborn),"Let It Go - From ""Frozen""/Soundtrack Version"
master_metadata_album_artist_name,Lorne Balfe,Idina Menzel
master_metadata_album_album_name,The Wheel of Time: The First Turn (Amazon Orig...,Frozen


In [8]:
df3.head(2).T

,0,1
ts,2022-12-01T04:27:21Z,2022-12-14T07:38:07Z
username,392uowv8fh00b9ie6ccn9cegs,392uowv8fh00b9ie6ccn9cegs
platform,"iOS 16.1.1 (iPad7,5)","Android OS 12 API 31 (motorola, moto g(60))"
ms_played,0,0
conn_country,IN,IN
ip_addr_decrypted,103.27.9.102,103.27.9.110
user_agent_decrypted,unknown,unknown
master_metadata_track_name,NaN,NaN
master_metadata_album_artist_name,NaN,NaN
master_metadata_album_album_name,NaN,NaN


## Some Observations

In [9]:
print(f"There are {len(df0)} rows in df0")
print(f"There are {len(df1)} rows in df1")
print(f"There are {len(df2)} rows in df2")
print(f"There are {len(df3)} rows in df3")
total = len(df0) + len(df1) + len(df2) + len(df3)
print(f"There are {total} rows in total")

There are 15657 rows in df0
There are 15656 rows in df1
There are 12132 rows in df2
There are 39 rows in df3
There are 43484 rows in total


Here are some observations:

1. The rows are not sorted by date.
2. `df3` has a lot of nulll values. We might need to get rid of them.
3. There are a number of `reason_start` and `reason_end`.
4. A number of extra columns are there which we don't need.

# Preprocessing

Here are the steps we'll follow to preprocess the data:
 
0. Some preliminary Analysis.
1. Remove unnecessary columns and rename columns.
2. Convert `ts` to datetime, change to timzone to IST.
3. Convert `offline_timestamp` to datetime.
4. Merge the dataframes and sort by `ts`.
5. Deal With Missing Values

## Some Preliminary Analysis

### Creating Some Functions

As I was doing some preliminary analysis, I realized that I was repeating the same code over and over again. So, I decided to create some functions to make my life easier.

In [10]:
color_codes = {
    "green": "\033[92m",
    "warning": "\033[93m",
    "fail": "\033[91m",
    "end": "\033[0m",
    "header": "\033[95m",
    "bold": "\033[1m",
    "underline": "\033[4m",
    "default": "\033[99m",
}

def pprint(text, color_code="default", *args, **kwargs):
    print(color_codes[color_code] + text + color_codes["end"], *args, **kwargs)

In [11]:
def analyze_column(column):
    pprint(f"Number of unique values:", "header")
    pprint("-----------------"*4, "green")
    pprint(f"Number of unique values in df0: {df0[column].nunique()}")
    pprint(f"Number of unique values in df1: {df1[column].nunique()}")
    pprint(f"Number of unique values in df2: {df2[column].nunique()}")
    pprint(f"Number of unique values in df3: {df3[column].nunique()}")

    col_uniques_0 = df0[column].unique()
    col_uniques_1 = df1[column].unique()
    col_uniques_2 = df2[column].unique()
    col_uniques_3 = df3[column].unique()
    col_all_uniques = np.concatenate([col_uniques_0, col_uniques_1, col_uniques_2, col_uniques_3])
    col_all_uniques = np.unique(col_all_uniques)

    pprint(f"\nNumber of unique values in all dataframes: {len(col_all_uniques)}")

    pprint(f"\nHere are the unique values in the {column}", color_code="header")
    pprint("-----------------"*4, "green")
    if len(col_all_uniques)>50:
        col_all_uniques = col_uniques_0[:50]
    pprint(" || ".join(col_all_uniques))


### `username` Column

In [12]:
df0["username"].nunique(), df1["username"].nunique(), df2["username"].nunique(), df3["username"].nunique()

(1, 1, 1, 1)

Great! This should be the case.

### `platform` Column

In [13]:
df0["platform"].nunique(), df1["platform"].nunique(), df2["platform"].nunique(), df3["platform"].nunique()

(21, 20, 21, 4)

That's interesting. I don't remeber listening on that many platforms. Let's see what they are.

In [14]:
analyze_column("platform")

Number of unique values:
--------------------------------------------------------------------
Number of unique values in df0: 21
Number of unique values in df1: 20
Number of unique values in df2: 21
Number of unique values in df3: 4

Number of unique values in all dataframes: 22

Here are the unique values in the platform
--------------------------------------------------------------------
Android OS 10 API 29 (Xiaomi, Redmi Note 7S) || Android OS 11 API 30 (motorola, moto g(60)) || Android OS 12 API 31 (motorola, moto g(60)) || Linux Ubuntu Core 18 (snap package) [x86-64 0] || Windows 10 (10.0.18363; x64; AppX) || Windows 10 (10.0.19042; x64; AppX) || Windows 10 (10.0.19043; x64; AppX) || Windows 10 (10.0.22000; x64; AppX) || Windows 10 (10.0.22621; x64; AppX) || iOS 14.3 (iPad7,5) || iOS 14.4 (iPad7,5) || iOS 14.4.1 (iPad7,5) || iOS 14.6 (iPad7,5) || iOS 14.7.1 (iPad7,5) || iOS 15.0 (iPad7,5) || iOS 15.0.1 (iPad7,5) || iOS 15.1 (iPad7,5) || iOS 15.4.1 (iPad7,5) || iOS 15.5 (iPad7,5) 

This makes sense. But where is the Windows 11? I've used it. Anyway, let's move on.

### The `conn_country` Column

In [15]:
df0["conn_country"].nunique(), df1["conn_country"].nunique(), df2["conn_country"].nunique(), df3["conn_country"].nunique()

(3, 5, 4, 1)

What? At least five countries? I don't think so. Let's see what they are. But before, we need to have a mapping between the country codes and the country names.

In [16]:
country_code = pd.read_html("https://www.worlddata.info/countrycodes.php")[0]
country_code = country_code[["Country", "ISO 3166-1 alpha2"]]
country_code.columns = ["Country", "Code"]
country_code.head()

,Country,Code
0,Afghanistan,AF
1,Åland Islands,AX
2,Albania,AL
3,Algeria,DZ
4,American Samoa,AS


In [17]:
countries0 = df0["conn_country"].unique()
countries1 = df1["conn_country"].unique()
countries2 = df2["conn_country"].unique()
countries3 = df3["conn_country"].unique()

countries = np.concatenate([countries0, countries1, countries2, countries3])
countries = np.unique(countries)

print("There are", len(countries), "unique countries")
for country in countries:
    c = country_code[country_code["Code"] == country]
    if len(c) > 0:
        print(c.iloc[0, 0], end=" || ")
    else:
        print(country, end=" || ")

There are 5 unique countries
Canada || Germany || United Kingdom || India || United States of America || 

To be honest, I've no idea when I was in these countries :). Let's move on.

### The `ip_addr_decrypted` Column

In [18]:
analyze_column("ip_addr_decrypted")

Number of unique values:
--------------------------------------------------------------------
Number of unique values in df0: 2221
Number of unique values in df1: 2205
Number of unique values in df2: 1959
Number of unique values in df3: 5

Number of unique values in all dataframes: 3071

Here are the unique values in the ip_addr_decrypted
--------------------------------------------------------------------
47.15.125.198 || 202.142.121.72 || 103.27.9.107 || 157.37.222.238 || 103.27.9.106 || 47.31.99.106 || 47.15.9.74 || 203.81.243.233 || 103.55.72.30 || 157.37.205.90 || 203.81.243.134 || 103.27.9.102 || 157.37.208.102 || 103.27.9.104 || 47.15.159.197 || 47.15.199.85 || 47.9.111.252 || 47.31.128.168 || 103.27.9.103 || 157.37.202.226 || 103.27.9.110 || 47.15.208.74 || 117.211.192.36 || 47.9.74.205 || 47.8.38.227 || 47.9.96.24 || 47.8.29.233 || 47.9.191.60 || 169.197.127.98 || 203.81.242.47 || 203.81.240.198 || 47.31.255.212 || 8.17.206.100 || 47.9.98.104 || 162.221.195.133 || 157.37.162.2

They are so many!

### The `user_agent_decrypted` Column

In [19]:
analyze_column("user_agent_decrypted")

Number of unique values:
--------------------------------------------------------------------
Number of unique values in df0: 1
Number of unique values in df1: 1
Number of unique values in df2: 1
Number of unique values in df3: 1

Number of unique values in all dataframes: 1

Here are the unique values in the user_agent_decrypted
--------------------------------------------------------------------
unknown


Hmm.

### The `reason_start` Column

I'm skipping some columns.

In [20]:
analyze_column("reason_start")

Number of unique values:
--------------------------------------------------------------------
Number of unique values in df0: 9
Number of unique values in df1: 8
Number of unique values in df2: 8
Number of unique values in df3: 3

Number of unique values in all dataframes: 10

Here are the unique values in the reason_start
--------------------------------------------------------------------
 || appload || backbtn || clickrow || fwdbtn || playbtn || remote || trackdone || trackerror || unknown


These ten are the different ways, a song was started.

### The `reason_end` Column

In [21]:
analyze_column("reason_end")

Number of unique values:
--------------------------------------------------------------------
Number of unique values in df0: 10
Number of unique values in df1: 10
Number of unique values in df2: 10
Number of unique values in df3: 3

Number of unique values in all dataframes: 10

Here are the unique values in the reason_end
--------------------------------------------------------------------
backbtn || endplay || fwdbtn || logout || remote || trackdone || trackerror || unexpected-exit || unexpected-exit-while-paused || unknown


These ten are the different ways, a song was ended.

Remaining columns are boolean, so skipped.

## Removing Unnecessary Columns

The columns we'll keep are:
```python
['ts', 'platform', 'ms_played', 'conn_country', 'ip_addr_decrypted', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start', 'reason_end', 'shuffle', 'offline', 'offline_timestamp']
```

In [109]:
columns_to_keep =['ts', 'platform', 'ms_played', 'conn_country', 'ip_addr_decrypted', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start', 'reason_end', 'shuffle',  'offline', 'offline_timestamp']

In [110]:
df0_f = df0[columns_to_keep]
df1_f = df1[columns_to_keep]
df2_f = df2[columns_to_keep]
df3_f = df3[columns_to_keep]

print(f"There are {len(df0_f)} rows in df0_f")
print(f"There are {len(df1_f)} rows in df1_f")
print(f"There are {len(df2_f)} rows in df2_f")
print(f"There are {len(df3_f)} rows in df3_f")

There are 15657 rows in df0_f
There are 15656 rows in df1_f
There are 12132 rows in df2_f
There are 39 rows in df3_f


We'll also rename some columns.

In [112]:
renamed_columns = ["Time", "Platform", "ms Played", "Country", "IP Address", "Track Name", "Artist Name", "Album Name", "Spotify URI", "Reason Start", "Reason End", "Shuffle", "Offline", "Offline Timestamp"]

df0_f.columns = renamed_columns
df1_f.columns = renamed_columns
df2_f.columns = renamed_columns
df3_f.columns = renamed_columns

## Time Index

### The `Time` Column

In [113]:
df0_f["Time"] = pd.to_datetime(df0_f["Time"])
df1_f["Time"] = pd.to_datetime(df1_f["Time"])
df2_f["Time"] = pd.to_datetime(df2_f["Time"])
df3_f["Time"] = pd.to_datetime(df3_f["Time"])

df0_f.set_index("Time", inplace=True)
df1_f.set_index("Time", inplace=True)
df2_f.set_index("Time", inplace=True)
df3_f.set_index("Time", inplace=True)

df0_f.index = df0_f.index.tz_convert("Asia/Kolkata")
df1_f.index = df1_f.index.tz_convert("Asia/Kolkata")
df2_f.index = df2_f.index.tz_convert("Asia/Kolkata")
df3_f.index = df3_f.index.tz_convert("Asia/Kolkata")

In [114]:
df0_f.head(2).T

Time,2022-02-25 11:41:22+05:30,2022-02-15 15:26:23+05:30
Platform,"Android OS 10 API 29 (Xiaomi, Redmi Note 7S)","iOS 15.1 (iPad7,5)"
ms Played,238866,211160
Country,IN,IN
IP Address,47.15.125.198,202.142.121.72
Track Name,Begin Again (Taylor's Version),Oops!...I Did It Again
Artist Name,Taylor Swift,Britney Spears
Album Name,Red (Taylor's Version),Oops!... I Did It Again
Spotify URI,spotify:track:05GsNucq8Bngd9fnd4fRa0,spotify:track:6naxalmIoLFWR0siv8dnQQ
Reason Start,trackdone,trackdone
Reason End,trackdone,trackdone


### The `Offline Timestamp` Column

Converting timestamp to date and then setting the timezone to IST was not working at first. I have to use the index of the new datetime for this to work.

In [115]:
df0_f["Offline Timestamp"] = pd.to_datetime(df0_f["Offline Timestamp"], unit="ms").tz_convert("Asia/Kolkata").index
df1_f["Offline Timestamp"] = pd.to_datetime(df1_f["Offline Timestamp"], unit="ms").tz_convert("Asia/Kolkata").index
df2_f["Offline Timestamp"] = pd.to_datetime(df2_f["Offline Timestamp"], unit="ms").tz_convert("Asia/Kolkata").index
df3_f["Offline Timestamp"] = pd.to_datetime(df3_f["Offline Timestamp"], unit="ms").tz_convert("Asia/Kolkata").index

In [116]:
df2_f.head(2).T

Time,2021-11-12 13:41:23+05:30,2021-10-27 11:41:21+05:30
Platform,"iOS 15.0.1 (iPad7,5)","Android OS 10 API 29 (Xiaomi, Redmi Note 7S)"
ms Played,136695,51292
Country,IN,IN
IP Address,202.142.122.89,203.81.240.198
Track Name,Aman Syndai (Dragon Reborn),"Let It Go - From ""Frozen""/Soundtrack Version"
Artist Name,Lorne Balfe,Idina Menzel
Album Name,The Wheel of Time: The First Turn (Amazon Orig...,Frozen
Spotify URI,spotify:track:4pjY12UohDZKJbkXWRa8u5,spotify:track:0qcr5FMsEO85NAQjrlDRKo
Reason Start,clickrow,trackdone
Reason End,logout,logout


## Merge the Dataframes

In [117]:
df = pd.concat([df0_f, df1_f, df2_f, df3_f])

In [125]:
df = df.sort_index()

## Missing Values

In [126]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 43484 entries, 2021-01-06 18:41:06+05:30 to 2023-01-05 21:07:28+05:30
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype                       
---  ------             --------------  -----                       
 0   Platform           43484 non-null  object                      
 1   ms Played          43484 non-null  int64                       
 2   Country            43484 non-null  object                      
 3   IP Address         43484 non-null  object                      
 4   Track Name         43442 non-null  object                      
 5   Artist Name        43442 non-null  object                      
 6   Album Name         43442 non-null  object                      
 7   Spotify URI        43442 non-null  object                      
 8   Reason Start       43484 non-null  object                      
 9   Reason End         43484 non-null  object                      
 10  Shuffle    

There are very few null values.

In [128]:
df.isna().sum()

Platform              0
ms Played             0
Country               0
IP Address            0
Track Name           42
Artist Name          42
Album Name           42
Spotify URI          42
Reason Start          0
Reason End            0
Shuffle               0
Offline              39
Offline Timestamp     0
dtype: int64

Let's have a look at some of them

In [129]:
df[df["Track Name"].isna()]

,Platform,ms Played,Country,IP Address,Track Name,Artist Name,Album Name,Spotify URI,Reason Start,Reason End,Shuffle,Offline,Offline Timestamp
Time,,,,,,,,,,,,,
2021-01-26 17:11:38+05:30,"iOS 14.3 (iPad7,5)",111966,IN,47.9.123.43,None,None,None,None,clickrow,endplay,False,0.0,2021-01-26 17:11:38+05:30
2021-01-26 20:26:43+05:30,"iOS 14.3 (iPad7,5)",45650,IN,47.9.123.43,None,None,None,None,clickrow,logout,False,0.0,2021-01-26 20:26:43+05:30
2021-05-25 07:35:17+05:30,"Android OS 10 API 29 (Xiaomi, Redmi Note 7S)",46833,IN,47.8.30.192,None,None,None,None,clickrow,endplay,False,0.0,2021-05-25 07:35:17+05:30
2021-12-03 12:21:09+05:30,"Android OS 10 API 29 (Xiaomi, Redmi Note 7S)",5803,IN,47.15.4.155,NaN,NaN,NaN,NaN,trackdone,trackdone,False,NaN,2021-12-03 12:21:09+05:30
2021-12-04 20:05:25+05:30,Windows 10 (10.0.22000; x64; AppX),1984,IN,203.81.240.212,NaN,NaN,NaN,NaN,trackdone,trackerror,False,NaN,2021-12-04 20:05:25+05:30
2021-12-04 20:34:06+05:30,Windows 10 (10.0.22000; x64; AppX),1489,IN,203.81.240.212,NaN,NaN,NaN,NaN,trackdone,trackerror,False,NaN,2021-12-04 20:34:06+05:30
2021-12-04 21:15:16+05:30,Windows 10 (10.0.22000; x64; AppX),945,IN,203.81.240.212,NaN,NaN,NaN,NaN,trackdone,trackerror,False,NaN,2021-12-04 21:15:16+05:30
2021-12-19 11:49:47+05:30,"Android OS 10 API 29 (Xiaomi, Redmi Note 7S)",5807,IN,202.142.122.54,NaN,NaN,NaN,NaN,trackdone,trackdone,True,NaN,2021-12-19 11:49:47+05:30
2022-12-01 09:56:20+05:30,"iOS 16.1.1 (iPad7,5)",16533,IN,103.27.9.102,NaN,NaN,NaN,NaN,unknown,trackdone,False,NaN,2022-12-01 09:56:20+05:30


We'll just drop them. They are not that much anyway!

In [134]:
df.dropna(inplace=True)

In [137]:
df.head(3).T

Time,2021-01-06 18:41:06+05:30,2021-01-06 18:46:42+05:30,2021-01-06 18:47:52+05:30
Platform,"Android OS 10 API 29 (Xiaomi, Redmi Note 7S)","Android OS 10 API 29 (Xiaomi, Redmi Note 7S)","Android OS 10 API 29 (Xiaomi, Redmi Note 7S)"
ms Played,42512,251,19600
Country,IN,IN,IN
IP Address,47.8.102.145,47.8.102.145,47.8.102.145
Track Name,exile (feat. Bon Iver),willow - dancing witch version (Elvira remix),willow
Artist Name,Taylor Swift,Taylor Swift,Taylor Swift
Album Name,folklore,willow,evermore
Spotify URI,spotify:track:4pvb0WLRcMtbPGmtejJJ6y,spotify:track:2NnrAdjE9cPdMklonMBuAv,spotify:track:0lx2cLdOt3piJbcaXIV74f
Reason Start,clickrow,clickrow,clickrow
Reason End,endplay,endplay,endplay


In [136]:
df.tail(2).T

Time,2023-01-05 21:02:11+05:30,2023-01-05 21:07:28+05:30
Platform,"Android OS 12 API 31 (motorola, moto g(60))","iOS 16.1.1 (iPad7,5)"
ms Played,328866,48251
Country,IN,IN
IP Address,103.27.9.108,103.27.9.105
Track Name,Bhor Bhaye Panghat Pe,Sanon Rog Laan Waliya
Artist Name,Lata Mangeshkar,Nusrat Fateh Ali Khan
Album Name,Satyam Shivam Sundaram,Nusrat Fateh Ali Khan No. 1 Hits
Spotify URI,spotify:track:2zIZUisgDGXPD3pvORaMi4,spotify:track:17aukIVZHitvQEWylFm92d
Reason Start,trackdone,clickrow
Reason End,trackdone,logout


Let's save this.

In [147]:
df.to_csv(os.path.join(data_dir, "cleaned.csv"))

In [139]:
df.shape

(43442, 13)

We'll also remove thoe songs which have been played less than half a minute.

In [148]:
df_final = df[df["ms Played"]>30000]

In [149]:
df_final.to_csv(os.path.join(data_dir, "final.csv"))